# Five-Fold MIL Pipeline

This notebook orchestrates the paper-aligned five-fold workflow now integrated into `ai/03_mil_training`. The canonical feature index is `/data/feature_extraction/feat_index.csv`.

1. Create balanced five-fold splits.
2. Train six modality-specific MIL models for every fold.
3. Evaluate the six-modality ensemble for every fold.
4. Aggregate final validation statistics.

In [ ]:
from pathlib import Path
import subprocess
import sys

TRAINING_DIR = Path.cwd().resolve()
if TRAINING_DIR.name != "03_mil_training":
    candidate = TRAINING_DIR / "ai" / "03_mil_training"
    if candidate.is_dir():
        TRAINING_DIR = candidate
    else:
        raise FileNotFoundError("Run this notebook from the repository root or ai/03_mil_training.")

def run_script(name, *arguments):
    command = [sys.executable, str(TRAINING_DIR / name), *map(str, arguments)]
    print(" ".join(command))
    subprocess.run(command, cwd=TRAINING_DIR, check=True)

TRAINING_DIR

## Step 1: Create Five-Fold Splits

In [ ]:
run_script("10_make_5fold_split.py")

## Step 2: Train Fold Models

In [ ]:
RUN_TRAINING = False
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    run_script("11_run_5fold_training.py", "--folds", "1", "--only", "vision", "--epochs", "1", "--num-workers", "0", "--skip-existing")
elif RUN_TRAINING:
    run_script("11_run_5fold_training.py", "--skip-existing")
else:
    print("Training is guarded. Set RUN_TRAINING=True or RUN_SMOKE_TEST=True to execute it.")

## Step 3: Evaluate Fold Ensembles

In [ ]:
RUN_ENSEMBLE = False
if RUN_ENSEMBLE:
    run_script("12_run_5fold_ensemble.py", "--skip-existing")
else:
    print("Ensemble evaluation is guarded. Set RUN_ENSEMBLE=True after training completes.")

## Step 4: Aggregate Final Validation Results

In [ ]:
run_script("13_final_validation.py")